# 016 Joint Decoding–Clustering Score

This notebook explicitly tests whether particular model orders and top-\(m\) archetype subsets jointly optimize:

1. decoding performance, and
2. condition-relevant clustering structure.

It is designed to support or revise statements like:

> The strongest joint decoding--clustering structure emerged at intermediate spatial model orders such as \(K=50\).

The notebook uses the summary CSV produced by notebook 014:

```python
spatial_enabled_clustering_validation_summary_across.csv
```

For each row, it computes:

\[
\mathrm{JointScore} = z(\mathrm{decode}) + z(\mathrm{clustering\ advantage})
\]

where clustering advantage is the observed top-decoding subset clustering metric minus the mean of random archetype subsets from the same model and \(K\).


In [ ]:
# ============================================================
# SETTINGS
# ============================================================

FIT_SCOPE = "across"

FIG_ROOT = "/Users/lowen/Desktop/papers/archetypes/figures"
NOTEBOOK14_DIR = "014_spatial_enabled_top_decoding_clustering_validation"
SUMMARY_CSV = (
    f"{FIG_ROOT}/{NOTEBOOK14_DIR}/"
    f"spatial_enabled_clustering_validation_summary_{FIT_SCOPE}.csv"
)

FIG_NOTEBOOK_DIR = "016_joint_decoding_clustering_score"
SAVE_FIGS = True
FIG_FORMAT = "pdf"
DPI = 300
FIGSIZE = (8.5, 5.2)

CLUSTER_METRICS = [
    "nmi",
    "ari",
    "purity",
    "nearest_centroid_accuracy",
    "silhouette_true_labels",
]

PRIMARY_CLUSTER_METRIC = "nmi"

CONDITIONS = ["intact", "word", "rest"]
TOP_M_VALUES = [1, 3, 5, 7, 10, 15, 20, 25]

SPATIAL_DENOMINATOR = 700
TEMPORAL_DENOMINATOR = 300

# Options: "condition_metric", "condition_metric_analysis", "global_metric"
NORMALIZE_WITHIN = "condition_metric"

DECODE_WEIGHT = 1.0
CLUSTER_WEIGHT = 1.0

PLOT_CONDITIONS = ["intact", "word", "rest"]
PLOT_TOP_M = [5, 10, 20]


In [ ]:
# ============================================================
# IMPORTS AND HELPERS
# ============================================================

%matplotlib inline

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG_DIR = Path(FIG_ROOT) / FIG_NOTEBOOK_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

_fig_counter = 0

def _safe_name(name):
    name = str(name).replace(" ", "_").replace("/", "-").replace("|", "_")
    name = "".join(ch for ch in name if ch.isalnum() or ch in ["_", "-", "."])
    return name[:180] if name else "figure"

def save_current_fig(name):
    global _fig_counter
    if not SAVE_FIGS:
        return None
    _fig_counter += 1
    out = FIG_DIR / f"{_fig_counter:03d}_{_safe_name(name)}.{FIG_FORMAT}"
    plt.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

def zscore_safe(x):
    x = np.asarray(x, dtype=float)
    mu = np.nanmean(x)
    sd = np.nanstd(x)
    if not np.isfinite(sd) or sd < 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - mu) / sd

def percentile_rank_safe(x):
    return pd.Series(x, dtype=float).rank(pct=True, method="average").to_numpy()

def component_ratio(K, analysis_type):
    return float(K) / (SPATIAL_DENOMINATOR if analysis_type == "spatial" else TEMPORAL_DENOMINATOR)

print("Figure directory:", FIG_DIR)


In [ ]:
# ============================================================
# LOAD NOTEBOOK 014 SUMMARY
# ============================================================

if "clustering_validation_summary_df" in globals():
    summary_df = clustering_validation_summary_df.copy()
    print("Using existing clustering_validation_summary_df from memory.")
else:
    path = Path(SUMMARY_CSV)
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Run notebook 014 first, or update SUMMARY_CSV."
        )
    summary_df = pd.read_csv(path)
    print("Loaded:", path)

summary_df["analysis_type"] = summary_df["analysis_type"].astype(str)
summary_df["condition"] = summary_df["condition"].astype(str)
summary_df["K"] = summary_df["K"].astype(int)
summary_df["top_m"] = summary_df["top_m"].astype(int)

if "component_ratio" not in summary_df.columns:
    summary_df["component_ratio"] = summary_df.apply(
        lambda r: component_ratio(r["K"], r["analysis_type"]), axis=1
    )

summary_df = summary_df[summary_df["top_m"].isin(TOP_M_VALUES)].copy()

print("Rows:", len(summary_df))
display(summary_df.head())


In [ ]:
# ============================================================
# BUILD LONG JOINT-SCORE TABLE
# ============================================================

rows = []

for metric in CLUSTER_METRICS:
    obs_col = f"{metric}_observed"
    rand_col = f"{metric}_random_mean"
    diff_col = f"{metric}_observed_minus_random"
    z_col = f"{metric}_z"
    p_col = f"{metric}_p_high"

    missing = [c for c in [obs_col, rand_col, diff_col, z_col] if c not in summary_df.columns]
    if missing:
        print(f"Skipping {metric}; missing {missing}")
        continue

    tmp = summary_df.copy()
    tmp["cluster_metric"] = metric
    tmp["cluster_observed"] = tmp[obs_col]
    tmp["cluster_random_mean"] = tmp[rand_col]
    tmp["cluster_observed_minus_random"] = tmp[diff_col]
    tmp["cluster_z_vs_random"] = tmp[z_col]
    tmp["cluster_p_high"] = tmp[p_col] if p_col in tmp.columns else np.nan
    rows.append(tmp)

joint_df = pd.concat(rows, ignore_index=True)

if NORMALIZE_WITHIN == "condition_metric":
    group_cols = ["condition", "cluster_metric"]
elif NORMALIZE_WITHIN == "condition_metric_analysis":
    group_cols = ["condition", "cluster_metric", "analysis_type"]
elif NORMALIZE_WITHIN == "global_metric":
    group_cols = ["cluster_metric"]
else:
    raise ValueError("Unknown NORMALIZE_WITHIN option.")

joint_df["decode_z"] = np.nan
joint_df["cluster_effect_z"] = np.nan
joint_df["decode_percentile"] = np.nan
joint_df["cluster_percentile"] = np.nan

for _, idx in joint_df.groupby(group_cols).groups.items():
    idx = list(idx)
    joint_df.loc[idx, "decode_z"] = zscore_safe(joint_df.loc[idx, "decode_mean"])
    joint_df.loc[idx, "cluster_effect_z"] = zscore_safe(
        joint_df.loc[idx, "cluster_observed_minus_random"]
    )
    joint_df.loc[idx, "decode_percentile"] = percentile_rank_safe(joint_df.loc[idx, "decode_mean"])
    joint_df.loc[idx, "cluster_percentile"] = percentile_rank_safe(
        joint_df.loc[idx, "cluster_observed_minus_random"]
    )

joint_df["joint_zsum"] = (
    DECODE_WEIGHT * joint_df["decode_z"] +
    CLUSTER_WEIGHT * joint_df["cluster_effect_z"]
)

joint_df["joint_percentile_geom"] = np.sqrt(
    np.clip(joint_df["decode_percentile"], 0, 1) *
    np.clip(joint_df["cluster_percentile"], 0, 1)
)

joint_df["joint_percentile_product"] = (
    np.clip(joint_df["decode_percentile"], 0, 1) *
    np.clip(joint_df["cluster_percentile"], 0, 1)
)

joint_df["joint_zsum_random_validated"] = joint_df["joint_zsum"].where(
    joint_df["cluster_z_vs_random"] > 0,
    np.nan,
)

print("Joint rows:", len(joint_df))
display(
    joint_df.sort_values("joint_zsum", ascending=False).head(30)[[
        "analysis_type", "condition", "K", "component_ratio", "top_m",
        "cluster_metric", "decode_mean", "cluster_observed_minus_random",
        "cluster_z_vs_random", "decode_z", "cluster_effect_z",
        "joint_zsum", "joint_percentile_geom"
    ]]
)


In [ ]:
# ============================================================
# TOP JOINT CASES
# ============================================================

def show_top_joint_cases(
    df,
    metric=PRIMARY_CLUSTER_METRIC,
    condition=None,
    analysis_type=None,
    top_n=25,
    score_col="joint_zsum",
):
    sub = df[df["cluster_metric"] == metric].copy()
    if condition is not None:
        sub = sub[sub["condition"] == condition].copy()
    if analysis_type is not None:
        sub = sub[sub["analysis_type"] == analysis_type].copy()

    keep = [
        "analysis_type", "condition", "K", "component_ratio", "top_m",
        "decode_mean", "decode_err",
        "cluster_observed", "cluster_random_mean",
        "cluster_observed_minus_random", "cluster_z_vs_random",
        "cluster_p_high",
        "decode_z", "cluster_effect_z",
        "decode_percentile", "cluster_percentile",
        "joint_zsum", "joint_percentile_geom",
    ]

    return sub.sort_values(score_col, ascending=False)[[c for c in keep if c in sub.columns]].head(top_n)

for condition in CONDITIONS:
    print("\n\n==============================")
    print("Condition:", condition, "| metric:", PRIMARY_CLUSTER_METRIC)
    print("==============================")
    display(show_top_joint_cases(joint_df, metric=PRIMARY_CLUSTER_METRIC, condition=condition, top_n=20))

print("\n\nSpatial only")
display(show_top_joint_cases(joint_df, metric=PRIMARY_CLUSTER_METRIC, analysis_type="spatial", top_n=30))

print("\n\nTemporal only")
display(show_top_joint_cases(joint_df, metric=PRIMARY_CLUSTER_METRIC, analysis_type="temporal", top_n=30))


In [ ]:
# ============================================================
# INSPECT SPATIAL K=50 AND NEARBY REGIMES
# ============================================================

def inspect_regime(
    df,
    analysis_type="spatial",
    K_values=(35, 50, 70, 88, 100),
    metric=PRIMARY_CLUSTER_METRIC,
    condition=None,
):
    sub = df[
        (df["analysis_type"] == analysis_type) &
        (df["cluster_metric"] == metric) &
        (df["K"].isin(K_values))
    ].copy()

    if condition is not None:
        sub = sub[sub["condition"] == condition].copy()

    keep = [
        "analysis_type", "condition", "K", "component_ratio", "top_m",
        "decode_mean", "cluster_observed_minus_random",
        "cluster_z_vs_random",
        "decode_z", "cluster_effect_z", "joint_zsum",
        "joint_percentile_geom",
    ]

    return sub.sort_values(["condition", "joint_zsum"], ascending=[True, False])[
        [c for c in keep if c in sub.columns]
    ]

display(inspect_regime(
    joint_df,
    analysis_type="spatial",
    K_values=(35, 50, 70, 88, 100),
    metric=PRIMARY_CLUSTER_METRIC,
))


In [ ]:
# ============================================================
# PLOT JOINT SCORE ACROSS NORMALIZED COMPONENT RATIO
# ============================================================

def plot_joint_score_by_ratio(
    df,
    metric=PRIMARY_CLUSTER_METRIC,
    condition="intact",
    top_m=3,
    score_col="joint_zsum",
):
    sub = df[
        (df["cluster_metric"] == metric) &
        (df["condition"] == condition) &
        (df["top_m"] == top_m)
    ].copy()

    if len(sub) == 0:
        print("No rows:", metric, condition, top_m)
        return

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)

    for analysis_type, linestyle, marker in [("spatial", "-", "o"), ("temporal", "--", "s")]:
        a = sub[sub["analysis_type"] == analysis_type].sort_values("component_ratio")
        if len(a) == 0:
            continue

        ax.plot(
            a["component_ratio"],
            a[score_col],
            linestyle=linestyle,
            marker=marker,
            linewidth=2.2,
            label=analysis_type,
        )

        top = a.sort_values(score_col, ascending=False).head(3)
        for _, r in top.iterrows():
            ax.text(
                r["component_ratio"],
                r[score_col],
                f"K={int(r['K'])}",
                fontsize=8,
                ha="center",
                va="bottom",
            )

    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel(score_col)
    ax.set_title(f"{condition}: joint decoding–clustering score\nmetric={metric}, top-{top_m}")
    ax.legend(frameon=False)
    plt.tight_layout()
    save_current_fig(f"joint_score_by_ratio_{metric}_{condition}_top{top_m}_{score_col}")
    plt.show()
    plt.close()

for condition in PLOT_CONDITIONS:
    for top_m in PLOT_TOP_M:
        plot_joint_score_by_ratio(joint_df, metric=PRIMARY_CLUSTER_METRIC, condition=condition, top_m=top_m)


In [ ]:
# ============================================================
# SCATTER: DECODING VS CLUSTERING ADVANTAGE
# ============================================================

def plot_decode_vs_cluster(
    df,
    metric=PRIMARY_CLUSTER_METRIC,
    condition="intact",
    top_m=None,
):
    sub = df[(df["cluster_metric"] == metric) & (df["condition"] == condition)].copy()
    if top_m is not None:
        sub = sub[sub["top_m"] == top_m].copy()

    if len(sub) == 0:
        print("No rows.")
        return

    fig, ax = plt.subplots(figsize=(6.8, 5.5))

    for analysis_type, marker in [("spatial", "o"), ("temporal", "s")]:
        a = sub[sub["analysis_type"] == analysis_type].copy()
        if len(a) == 0:
            continue

        ax.scatter(
            a["decode_z"],
            a["cluster_effect_z"],
            s=40 + 8 * a["top_m"],
            alpha=0.75,
            marker=marker,
            label=analysis_type,
        )

        top = a.sort_values("joint_zsum", ascending=False).head(10)
        for _, r in top.iterrows():
            ax.text(
                r["decode_z"],
                r["cluster_effect_z"],
                f"K={int(r['K'])},m={int(r['top_m'])}",
                fontsize=7,
                ha="left",
                va="bottom",
            )

    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.axvline(0, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("Decoding z-score")
    ax.set_ylabel("Clustering-effect z-score")
    title = f"{condition}: decoding vs clustering effect\nmetric={metric}"
    if top_m is not None:
        title += f", top-{top_m}"
    ax.set_title(title)
    ax.legend(frameon=False)
    plt.tight_layout()
    save_current_fig(
        f"decode_vs_cluster_scatter_{metric}_{condition}_"
        f"{'alltopm' if top_m is None else 'top'+str(top_m)}"
    )
    plt.show()
    plt.close()

for condition in PLOT_CONDITIONS:
    plot_decode_vs_cluster(joint_df, metric=PRIMARY_CLUSTER_METRIC, condition=condition, top_m=None)
    for top_m in [3, 5, 10, 20, 50]:
        plot_decode_vs_cluster(joint_df, metric=PRIMARY_CLUSTER_METRIC, condition=condition, top_m=top_m)


In [ ]:
# ============================================================
# ENHANCED SCATTER
#
# marker = analysis type
# size   = top-m
# color  = ordered representational scale (K rank)
#
# Uses rank-based coloring so uneven K spacing
# does not distort the color mapping.
# ============================================================

def plot_decode_vs_cluster_enhanced(
    df,
    metric=PRIMARY_CLUSTER_METRIC,
    condition="intact",
    top_m=None,
    cmap="viridis",
    annotate_top_n=8,
):

    sub = df[
        (df["cluster_metric"] == metric)
        & (df["condition"] == condition)
    ].copy()

    if top_m is not None:
        sub = sub[sub["top_m"] == top_m].copy()

    if len(sub) == 0:
        print("No rows.")
        return

    # --------------------------------------------------------
    # Build rank-based K mapping
    # --------------------------------------------------------

    ALL_K = sorted(
        set(K_VALUES_BY_ANALYSIS["spatial"])
        | set(K_VALUES_BY_ANALYSIS["temporal"])
    )

    K_TO_RANK = {
        k: i
        for i, k in enumerate(ALL_K)
    }

    sub["K_rank"] = sub["K"].map(K_TO_RANK)

    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(8.2, 6.4)
    )

    marker_map = {
        "spatial": "o",
        "temporal": "s",
    }

    cmap_obj = plt.cm.get_cmap(cmap)

    norm = plt.Normalize(
        vmin=0,
        vmax=len(ALL_K) - 1
    )

    # --------------------------------------------------------
    # Plot by analysis type
    # --------------------------------------------------------

    for analysis_type in ["spatial", "temporal"]:

        a = sub[
            sub["analysis_type"] == analysis_type
        ].copy()

        if len(a) == 0:
            continue

        point_sizes = 40 + 18 * a["top_m"]

        sc = ax.scatter(
            a["decode_z"],
            a["cluster_effect_z"],
            c=a["K_rank"],
            cmap=cmap_obj,
            norm=norm,
            s=point_sizes,
            alpha=0.82,
            marker=marker_map[analysis_type],
            edgecolor="black",
            linewidth=0.5,
            label=analysis_type,
        )

        # ----------------------------------------------------
        # Annotate strongest joint cases
        # ----------------------------------------------------

        top = (
            a.sort_values(
                "joint_zsum",
                ascending=False
            )
            .head(annotate_top_n)
        )

        for _, r in top.iterrows():

            label = (
                f"K={int(r['K'])}\n"
                f"m={int(r['top_m'])}"
            )

            ax.text(
                r["decode_z"],
                r["cluster_effect_z"],
                label,
                fontsize=7,
                ha="left",
                va="bottom",
            )

    # --------------------------------------------------------
    # Guides
    # --------------------------------------------------------

    ax.axhline(
        0,
        color="gray",
        linestyle="--",
        linewidth=1,
    )

    ax.axvline(
        0,
        color="gray",
        linestyle="--",
        linewidth=1,
    )

    ax.set_xlabel(
        "Decoding z-score"
    )

    ax.set_ylabel(
        "Clustering-effect z-score"
    )

    title = (
        f"{condition}: decoding vs clustering structure\n"
        f"metric={metric}"
    )

    if top_m is not None:
        title += f" | top-{top_m}"

    ax.set_title(title)

    # --------------------------------------------------------
    # Analysis type legend
    # --------------------------------------------------------

    analysis_handles = []

    for analysis_type in ["spatial", "temporal"]:

        handle = plt.Line2D(
            [],
            [],
            linestyle="none",
            marker=marker_map[analysis_type],
            markersize=9,
            markeredgecolor="black",
            markerfacecolor="gray",
            label=analysis_type,
        )

        analysis_handles.append(handle)

    legend1 = ax.legend(
        handles=analysis_handles,
        title="Analysis type",
        loc="upper left",
        frameon=False,
    )

    ax.add_artist(legend1)

    # --------------------------------------------------------
    # Top-m legend
    # --------------------------------------------------------

    size_handles = []

    for m in sorted(
        sub["top_m"].unique()
    ):

        size_handles.append(
            plt.scatter(
                [],
                [],
                s=40 + 18 * m,
                color="gray",
                alpha=0.8,
                edgecolor="black",
                linewidth=0.5,
                label=f"top-{m}",
            )
        )

    legend2 = ax.legend(
        handles=size_handles,
        title="Subset size",
        loc="lower right",
        frameon=False,
    )

    ax.add_artist(legend2)

    # --------------------------------------------------------
    # Rank-based colorbar
    # --------------------------------------------------------

    sm = plt.cm.ScalarMappable(
        norm=norm,
        cmap=cmap_obj,
    )

    cbar = plt.colorbar(
        sm,
        ax=ax,
    )

    # show every ~3rd K value
    tick_step = max(
        1,
        len(ALL_K) // 10
    )

    tick_positions = np.arange(
        0,
        len(ALL_K),
        tick_step
    )

    cbar.set_ticks(
        tick_positions
    )

    cbar.set_ticklabels(
        [str(ALL_K[i]) for i in tick_positions]
    )

    cbar.set_label(
        "Representational scale (K)"
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    plt.tight_layout()

    save_current_fig(
        f"decode_vs_cluster_enhanced_rankK_"
        f"{metric}_{condition}_"
        f"{'alltopm' if top_m is None else 'top'+str(top_m)}"
    )

    plt.show()
    plt.close()

# ============================================================
# RUN
# ============================================================


K_VALUES_BY_ANALYSIS = {
    "spatial": [5, 7, 10, 14, 21, 25, 35, 40, 42, 44, 45, 46, 48, 50, 52, 53, 54, 55, 56, 58, 60, 63, 65, 70, 75, 88, 100, 105, 126, 140, 175, 189, 200, 210, 252, 300, 400, 500, 600, 700],
    "temporal": [2, 3, 5, 6, 8, 9, 10, 11, 14, 15, 17, 18, 20, 23, 25, 27, 30, 35, 38, 40, 45, 50, 54, 55, 60, 65, 75, 81, 90, 100, 108, 200, 300],
}

for condition in PLOT_CONDITIONS:

    # all top-m together
    plot_decode_vs_cluster_enhanced(
        joint_df,
        metric=PRIMARY_CLUSTER_METRIC,
        condition=condition,
        top_m=None,
        cmap="viridis",
        annotate_top_n=6,
    )

    # separate fixed top-m versions
    for top_m in [3, 5, 7, 10, 20]:

        plot_decode_vs_cluster_enhanced(
            joint_df,
            metric=PRIMARY_CLUSTER_METRIC,
            condition=condition,
            top_m=top_m,
            cmap="viridis",
            annotate_top_n=10,
        )

In [ ]:
# ============================================================
# TWO ENHANCED JOINT-SCORE SCATTERS
# 1) Spatial only: color = raw K, marker = top-m
# 2) Spatial/temporal: color = normalized component ratio,
#                      marker = analysis type, size = top-m
# ============================================================

from matplotlib.lines import Line2D

# ------------------------------------------------------------
# Shared helpers
# ------------------------------------------------------------

def _add_quadrant_guides(ax):
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.axvline(0, color="gray", linestyle="--", linewidth=1)


def _topm_marker_map(values):
    markers = ["o", "s", "^", "D", "P", "X", "v", "*"]
    vals = sorted(pd.unique(values))
    return {m: markers[i % len(markers)] for i, m in enumerate(vals)}


# ============================================================
# 1) SPATIAL ONLY
# color = raw K
# marker = top-m
# ============================================================

def plot_spatial_decode_vs_cluster_by_k_and_topm(
    df,
    metric=PRIMARY_CLUSTER_METRIC,
    condition="intact",
    cmap="viridis",
    annotate_top_n=8,
):
    sub = df[
        (df["cluster_metric"] == metric) &
        (df["condition"] == condition) &
        (df["analysis_type"] == "spatial")
    ].copy()

    if len(sub) == 0:
        print("No spatial rows.")
        return

    fig, ax = plt.subplots(figsize=(8.2, 6.4))

    marker_map = _topm_marker_map(sub["top_m"])

    kmin = sub["K"].min()
    kmax = sub["K"].max()
    norm = plt.Normalize(kmin, kmax)
    cmap_obj = plt.cm.get_cmap(cmap)

    for top_m, marker in marker_map.items():
        a = sub[sub["top_m"] == top_m].copy()

        sc = ax.scatter(
            a["decode_z"],
            a["cluster_effect_z"],
            c=a["K"],
            cmap=cmap_obj,
            norm=norm,
            s=115,
            alpha=0.85,
            marker=marker,
            edgecolor="black",
            linewidth=0.55,
            label=f"top-{top_m}",
        )

    # Label strongest joint cases
    top = sub.sort_values("joint_zsum", ascending=False).head(annotate_top_n)
    for _, r in top.iterrows():
        ax.text(
            r["decode_z"],
            r["cluster_effect_z"],
            f"K={int(r['K'])}\nm={int(r['top_m'])}",
            fontsize=7,
            ha="left",
            va="bottom",
        )

    _add_quadrant_guides(ax)

    ax.set_xlabel("Decoding z-score")
    ax.set_ylabel("Clustering-effect z-score")
    ax.set_title(
        f"Spatial AA only: decoding vs clustering structure\n"
        f"{condition}, metric={metric}; color=K, marker=top-m"
    )

    ax.legend(title="Subset size", frameon=False, loc="lower right")

    cbar = plt.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap_obj),
        ax=ax,
    )
    cbar.set_label("Raw component number K")

    plt.tight_layout()

    save_current_fig(
        f"spatial_decode_vs_cluster_colorK_markerTopM_{metric}_{condition}"
    )

    plt.show()
    plt.close()


# ============================================================
# 2) SPATIAL/TEMPORAL COMPARISON
# color = normalized component ratio
# marker = analysis type
# size = top-m
# ============================================================

def plot_decode_vs_cluster_by_ratio_analysis_topm(
    df,
    metric=PRIMARY_CLUSTER_METRIC,
    condition="intact",
    cmap="viridis",
    annotate_top_n=8,
):
    sub = df[
        (df["cluster_metric"] == metric) &
        (df["condition"] == condition)
    ].copy()

    if len(sub) == 0:
        print("No rows.")
        return

    fig, ax = plt.subplots(figsize=(8.4, 6.4))

    marker_map = {
        "spatial": "o",
        "temporal": "s",
    }

    rmin = sub["component_ratio"].min()
    rmax = sub["component_ratio"].max()
    norm = plt.Normalize(rmin, rmax)
    cmap_obj = plt.cm.get_cmap(cmap)

    for analysis_type, marker in marker_map.items():
        a = sub[sub["analysis_type"] == analysis_type].copy()

        if len(a) == 0:
            continue

        sizes = 45 + 22 * a["top_m"]

        ax.scatter(
            a["decode_z"],
            a["cluster_effect_z"],
            c=a["component_ratio"],
            cmap=cmap_obj,
            norm=norm,
            s=sizes,
            alpha=0.82,
            marker=marker,
            edgecolor="black",
            linewidth=0.55,
            label=analysis_type,
        )

    # Label strongest joint cases overall
    top = sub.sort_values("joint_zsum", ascending=False).head(annotate_top_n)

    for _, r in top.iterrows():
        ax.text(
            r["decode_z"],
            r["cluster_effect_z"],
            f"{r['analysis_type'][0].upper()} K={int(r['K'])}\n"
            f"r={r['component_ratio']:.2f}, m={int(r['top_m'])}",
            fontsize=7,
            ha="left",
            va="bottom",
        )

    _add_quadrant_guides(ax)

    ax.set_xlabel("Decoding z-score")
    ax.set_ylabel("Clustering-effect z-score")
    ax.set_title(
        f"Spatial vs temporal: decoding vs clustering structure\n"
        f"{condition}, metric={metric}; color=component ratio"
    )

    # Analysis-type legend
    analysis_handles = [
        Line2D(
            [],
            [],
            linestyle="none",
            marker=marker_map["spatial"],
            markersize=9,
            markeredgecolor="black",
            markerfacecolor="gray",
            label="spatial",
        ),
        Line2D(
            [],
            [],
            linestyle="none",
            marker=marker_map["temporal"],
            markersize=9,
            markeredgecolor="black",
            markerfacecolor="gray",
            label="temporal",
        ),
    ]

    legend1 = ax.legend(
        handles=analysis_handles,
        title="Analysis type",
        frameon=False,
        loc="upper left",
    )
    ax.add_artist(legend1)

    # top-m size legend
    size_handles = []
    for m in sorted(sub["top_m"].unique()):
        size_handles.append(
            plt.scatter(
                [],
                [],
                s=45 + 22 * m,
                color="gray",
                alpha=0.8,
                edgecolor="black",
                linewidth=0.55,
                label=f"top-{m}",
            )
        )

    legend2 = ax.legend(
        handles=size_handles,
        title="Subset size",
        frameon=False,
        loc="lower right",
    )
    ax.add_artist(legend2)

    cbar = plt.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap_obj),
        ax=ax,
    )
    cbar.set_label("Normalized component ratio")

    plt.tight_layout()

    save_current_fig(
        f"spatiotemporal_decode_vs_cluster_colorRatio_markerAnalysis_sizeTopM_"
        f"{metric}_{condition}"
    )

    plt.show()
    plt.close()


# ============================================================
# RUN
# ============================================================

for condition in PLOT_CONDITIONS:

    plot_spatial_decode_vs_cluster_by_k_and_topm(
        joint_df,
        metric=PRIMARY_CLUSTER_METRIC,
        condition=condition,
        cmap="viridis",
        annotate_top_n=8,
    )

    plot_decode_vs_cluster_by_ratio_analysis_topm(
        joint_df,
        metric=PRIMARY_CLUSTER_METRIC,
        condition=condition,
        cmap="viridis",
        annotate_top_n=8,
    )

In [ ]:
# ============================================================
# TWO ENHANCED JOINT-SCORE SCATTERS
#
# 1) Spatial only:
#    marker = top-m
#    color  = rank-ordered K
#
# 2) Spatial/temporal:
#    marker = analysis type
#    size   = top-m
#    color  = normalized component ratio
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ------------------------------------------------------------
# Shared helpers
# ------------------------------------------------------------

def _add_quadrant_guides(ax):
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.axvline(0, color="gray", linestyle="--", linewidth=1)


def _topm_marker_map(values):
    markers = ["o", "s", "^", "D", "P", "X", "v", "*"]
    vals = sorted(pd.unique(values))
    return {m: markers[i % len(markers)] for i, m in enumerate(vals)}


def _rank_k_values(df):
    all_k = sorted(df["K"].dropna().astype(int).unique())
    k_to_rank = {k: i for i, k in enumerate(all_k)}
    return all_k, k_to_rank


def _add_rank_k_colorbar(fig, ax, cmap_obj, norm, all_k, label="Representational scale (K)"):

    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap_obj)

    cbar = fig.colorbar(sm, ax=ax)

    tick_step = max(1, len(all_k) // 10)
    tick_positions = np.arange(0, len(all_k), tick_step)

    # make sure the final K value appears
    if tick_positions[-1] != len(all_k) - 1:
        tick_positions = np.append(tick_positions, len(all_k) - 1)

    cbar.set_ticks(tick_positions)
    cbar.set_ticklabels([str(all_k[i]) for i in tick_positions])
    cbar.set_label(label)

    return cbar


# ============================================================
# 1) SPATIAL ONLY
# color = rank-ordered K
# marker = top-m
# ============================================================

def plot_spatial_decode_vs_cluster_by_k_and_topm(
    df,
    metric=PRIMARY_CLUSTER_METRIC,
    condition="intact",
    cmap="viridis",
    annotate_top_n=8,
):
    sub = df[
        (df["cluster_metric"] == metric)
        & (df["condition"] == condition)
        & (df["analysis_type"] == "spatial")
    ].copy()

    if len(sub) == 0:
        print("No spatial rows.")
        return

    # rank-based K coloring
    all_k, k_to_rank = _rank_k_values(sub)
    sub["K_rank"] = sub["K"].astype(int).map(k_to_rank)

    fig, ax = plt.subplots(figsize=(8.2, 6.4))

    marker_map = _topm_marker_map(sub["top_m"])

    cmap_obj = plt.cm.get_cmap(cmap)
    norm = plt.Normalize(vmin=0, vmax=len(all_k) - 1)

    for top_m, marker in marker_map.items():

        a = sub[sub["top_m"] == top_m].copy()

        ax.scatter(
            a["decode_z"],
            a["cluster_effect_z"],
            c=a["K_rank"],
            cmap=cmap_obj,
            norm=norm,
            s=115,
            alpha=0.85,
            marker=marker,
            edgecolor="black",
            linewidth=0.55,
            label=f"top-{top_m}",
        )

    # Label strongest joint cases
    top = sub.sort_values("joint_zsum", ascending=False).head(annotate_top_n)

    for _, r in top.iterrows():
        ax.text(
            r["decode_z"],
            r["cluster_effect_z"],
            f"K={int(r['K'])}\nm={int(r['top_m'])}",
            fontsize=7,
            ha="left",
            va="bottom",
        )

    _add_quadrant_guides(ax)

    ax.set_xlabel("Decoding z-score")
    ax.set_ylabel("Clustering-effect z-score")

    ax.set_title(
        f"Spatial AA only: decoding vs clustering structure\n"
        f"{condition}, metric={metric}; color=ordered K, marker=top-m"
    )

    ax.legend(title="Subset size", frameon=False, loc="lower right")

    _add_rank_k_colorbar(
        fig,
        ax,
        cmap_obj,
        norm,
        all_k,
        label="Representational scale (K)",
    )

    plt.tight_layout()

    save_current_fig(
        f"spatial_decode_vs_cluster_rankK_markerTopM_{metric}_{condition}"
    )

    plt.show()
    plt.close()


# ============================================================
# 2) SPATIAL/TEMPORAL COMPARISON
# color = normalized component ratio
# marker = analysis type
# size = top-m
# ============================================================

def plot_decode_vs_cluster_by_ratio_analysis_topm(
    df,
    metric=PRIMARY_CLUSTER_METRIC,
    condition="intact",
    cmap="viridis",
    annotate_top_n=8,
):
    sub = df[
        (df["cluster_metric"] == metric)
        & (df["condition"] == condition)
    ].copy()

    if len(sub) == 0:
        print("No rows.")
        return

    fig, ax = plt.subplots(figsize=(8.4, 6.4))

    marker_map = {
        "spatial": "o",
        "temporal": "s",
    }

    rmin = sub["component_ratio"].min()
    rmax = sub["component_ratio"].max()

    norm = plt.Normalize(rmin, rmax)
    cmap_obj = plt.cm.get_cmap(cmap)

    for analysis_type, marker in marker_map.items():

        a = sub[sub["analysis_type"] == analysis_type].copy()

        if len(a) == 0:
            continue

        sizes = 45 + 22 * a["top_m"]

        ax.scatter(
            a["decode_z"],
            a["cluster_effect_z"],
            c=a["component_ratio"],
            cmap=cmap_obj,
            norm=norm,
            s=sizes,
            alpha=0.82,
            marker=marker,
            edgecolor="black",
            linewidth=0.55,
            label=analysis_type,
        )

    # Label strongest joint cases overall
    top = sub.sort_values("joint_zsum", ascending=False).head(annotate_top_n)

    for _, r in top.iterrows():
        ax.text(
            r["decode_z"],
            r["cluster_effect_z"],
            f"{r['analysis_type'][0].upper()} K={int(r['K'])}\n"
            f"r={r['component_ratio']:.2f}, m={int(r['top_m'])}",
            fontsize=7,
            ha="left",
            va="bottom",
        )

    _add_quadrant_guides(ax)

    ax.set_xlabel("Decoding z-score")
    ax.set_ylabel("Clustering-effect z-score")

    ax.set_title(
        f"Spatial vs temporal: decoding vs clustering structure\n"
        f"{condition}, metric={metric}; color=component ratio"
    )

    # Analysis-type legend
    analysis_handles = [
        Line2D(
            [],
            [],
            linestyle="none",
            marker=marker_map["spatial"],
            markersize=9,
            markeredgecolor="black",
            markerfacecolor="gray",
            label="spatial",
        ),
        Line2D(
            [],
            [],
            linestyle="none",
            marker=marker_map["temporal"],
            markersize=9,
            markeredgecolor="black",
            markerfacecolor="gray",
            label="temporal",
        ),
    ]

    legend1 = ax.legend(
        handles=analysis_handles,
        title="Analysis type",
        frameon=False,
        loc="upper left",
    )

    ax.add_artist(legend1)

    # top-m size legend
    size_handles = []

    for m in sorted(sub["top_m"].unique()):
        size_handles.append(
            plt.scatter(
                [],
                [],
                s=45 + 22 * m,
                color="gray",
                alpha=0.8,
                edgecolor="black",
                linewidth=0.55,
                label=f"top-{m}",
            )
        )

    legend2 = ax.legend(
        handles=size_handles,
        title="Subset size",
        frameon=False,
        loc="lower right",
    )

    ax.add_artist(legend2)

    cbar = plt.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap_obj),
        ax=ax,
    )

    cbar.set_label("Normalized component ratio")

    plt.tight_layout()

    save_current_fig(
        f"spatiotemporal_decode_vs_cluster_colorRatio_markerAnalysis_sizeTopM_"
        f"{metric}_{condition}"
    )

    plt.show()
    plt.close()


# ============================================================
# RUN
# ============================================================

for condition in PLOT_CONDITIONS:

    plot_spatial_decode_vs_cluster_by_k_and_topm(
        joint_df,
        metric=PRIMARY_CLUSTER_METRIC,
        condition=condition,
        cmap="viridis",
        annotate_top_n=8,
    )

    plot_decode_vs_cluster_by_ratio_analysis_topm(
        joint_df,
        metric=PRIMARY_CLUSTER_METRIC,
        condition=condition,
        cmap="viridis",
        annotate_top_n=8,
    )

In [ ]:
# ============================================================
# HEATMAP: K x TOP-M JOINT SCORE
# ============================================================

def plot_joint_heatmap(
    df,
    metric=PRIMARY_CLUSTER_METRIC,
    condition="intact",
    analysis_type="spatial",
    score_col="joint_zsum",
):
    sub = df[
        (df["cluster_metric"] == metric) &
        (df["condition"] == condition) &
        (df["analysis_type"] == analysis_type)
    ].copy()

    if len(sub) == 0:
        print("No rows.")
        return

    piv = sub.pivot_table(index="top_m", columns="K", values=score_col, aggfunc="mean")
    piv = piv.sort_index().reindex(sorted(piv.columns), axis=1)

    vmax = np.nanmax(np.abs(piv.values))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1

    fig, ax = plt.subplots(figsize=(9, 4.5))
    im = ax.imshow(piv.values, aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)

    ax.set_xticks(np.arange(len(piv.columns)))
    ax.set_xticklabels([str(c) for c in piv.columns], rotation=45, ha="right")
    ax.set_yticks(np.arange(len(piv.index)))
    ax.set_yticklabels([str(i) for i in piv.index])

    ax.set_xlabel("K")
    ax.set_ylabel("top-m")
    ax.set_title(f"{analysis_type} | {condition} | metric={metric}\n{score_col} across K and top-m")

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(score_col)

    plt.tight_layout()
    save_current_fig(f"joint_heatmap_{analysis_type}_{condition}_{metric}_{score_col}")
    plt.show()
    plt.close()

for analysis_type in ["spatial", "temporal"]:
    for condition in PLOT_CONDITIONS:
        plot_joint_heatmap(joint_df, metric=PRIMARY_CLUSTER_METRIC, condition=condition, analysis_type=analysis_type)


In [ ]:
# ============================================================
# CONSENSUS ACROSS CLUSTERING METRICS
# ============================================================

consensus_cols = ["analysis_type", "condition", "K", "component_ratio", "top_m"]

joint_consensus_df = (
    joint_df
    .groupby(consensus_cols, as_index=False)
    .agg(
        decode_mean=("decode_mean", "mean"),
        mean_joint_zsum=("joint_zsum", "mean"),
        median_joint_zsum=("joint_zsum", "median"),
        mean_joint_percentile=("joint_percentile_geom", "mean"),
        mean_cluster_z_vs_random=("cluster_z_vs_random", "mean"),
        min_cluster_p_high=("cluster_p_high", "min"),
    )
)

display(joint_consensus_df.sort_values("mean_joint_zsum", ascending=False).head(40))

def plot_consensus_joint_score(condition="intact", top_m=3, score_col="mean_joint_zsum"):
    sub = joint_consensus_df[
        (joint_consensus_df["condition"] == condition) &
        (joint_consensus_df["top_m"] == top_m)
    ].copy()

    if len(sub) == 0:
        return

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)

    for analysis_type, linestyle, marker in [("spatial", "-", "o"), ("temporal", "--", "s")]:
        a = sub[sub["analysis_type"] == analysis_type].sort_values("component_ratio")
        if len(a) == 0:
            continue

        ax.plot(
            a["component_ratio"],
            a[score_col],
            linestyle=linestyle,
            marker=marker,
            linewidth=2.2,
            label=analysis_type,
        )

        top = a.sort_values(score_col, ascending=False).head(3)
        for _, r in top.iterrows():
            ax.text(
                r["component_ratio"],
                r[score_col],
                f"K={int(r['K'])}",
                fontsize=8,
                ha="center",
                va="bottom",
            )

    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel(score_col)
    ax.set_title(f"{condition}: consensus joint score across clustering metrics | top-{top_m}")
    ax.legend(frameon=False)
    plt.tight_layout()
    save_current_fig(f"consensus_joint_score_{condition}_top{top_m}_{score_col}")
    plt.show()
    plt.close()

for condition in PLOT_CONDITIONS:
    for top_m in PLOT_TOP_M:
        plot_consensus_joint_score(condition=condition, top_m=top_m)


In [ ]:
# ============================================================
# NETWORK COMPOSITION OF BEST JOINT DECODING--CLUSTERING COMBINATIONS
#
# Append this to the end of notebook 016.
#
# Goal:
#   Among the best-performing K, top-m combinations from joint_df,
#   identify which Yeo networks are most represented in the selected
#   top-m archetype subsets.
# ============================================================

import ast
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

BEST_ANALYSIS_TYPE = "spatial"
BEST_CONDITION = "intact"
BEST_CLUSTER_METRIC = PRIMARY_CLUSTER_METRIC

BEST_TOP_N_CONFIGS = 25
BEST_MIN_TOP_M = 3
BEST_MAX_TOP_M = 15

MSAA_RESULTS_DIR = "."
DECODING_DIR_TEMPLATE = "msaa_condrank_decoding_outputs_{analysis_type}_{fit_scope}"

POSTERIOR_MAT = "data/pieman/raw/pieman_posterior_K700.mat"
SCHAEFER_TXT = "data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order.txt"
SCHAEFER_NII = "data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_2mm.nii.gz"

NETWORK_ORDER = [
    "Visual",
    "Somatomotor",
    "Dorsal attention",
    "Ventral attention",
    "Limbic",
    "Frontoparietal",
    "Default mode",
]

NETWORK_COLORS = {
    "Visual": "#D7DF23",
    "Somatomotor": "#39B54A",
    "Dorsal attention": "#00A79D",
    "Ventral attention": "#27AAE1",
    "Limbic": "#1C75BC",
    "Frontoparietal": "#92278F",
    "Default mode": "#EE2A7B",
}

LOOKUP_TABLE = {
    "Vis": "Visual",
    "SomMot": "Somatomotor",
    "DorsAttn": "Dorsal attention",
    "SalVentAttn": "Ventral attention",
    "Limbic": "Limbic",
    "Cont": "Frontoparietal",
    "Default": "Default mode",
}

NETWORK_CODES = {name: i + 1 for i, name in enumerate(LOOKUP_TABLE.values())}
NETWORK_NAME_MAP = {v: k for k, v in NETWORK_CODES.items()}

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def parse_selected_archetypes(x):
    if isinstance(x, list):
        return [int(v) for v in x]
    if isinstance(x, np.ndarray):
        return [int(v) for v in x.tolist()]
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            return [int(v) for v in ast.literal_eval(x)]
        except Exception:
            return [int(v) for v in re.findall(r"-?\d+", x)]
    return []


def find_msaa_npz(analysis_type, fit_scope, K, base_dir=MSAA_RESULTS_DIR):
    base_dir = Path(base_dir)
    candidates = []

    for p in base_dir.rglob("*.npz"):
        name = p.name.lower()
        if analysis_type.lower() in name and fit_scope.lower() in name and f"k{K}" in name:
            candidates.append(p)

    if len(candidates) == 0:
        for p in base_dir.rglob("*.npz"):
            name = p.name.lower()
            if analysis_type.lower() in name and f"k{K}" in name:
                candidates.append(p)

    if len(candidates) == 0:
        print(f"No npz found for {analysis_type} {fit_scope} K={K}")
        return None

    candidates = sorted(candidates, key=lambda p: (len(str(p)), str(p)))
    return candidates[0]


def load_npz_as_dict(path):
    z = np.load(path, allow_pickle=True)
    out = {}
    for key in z.files:
        val = z[key]
        if hasattr(val, "shape") and val.shape == () and val.dtype == object:
            val = val.item()
        out[key] = val
    return out


def unpack_results_subj(d):
    for key in ["results_subj", "results", "subject_results", "subj_results"]:
        if key in d:
            obj = d[key]
            if isinstance(obj, list):
                return obj
            if isinstance(obj, np.ndarray) and obj.dtype == object:
                return list(obj)

    if "sXC" in d and "S" in d:
        return [{"sXC": d["sXC"], "S": d["S"]}]

    for val in d.values():
        if isinstance(val, np.ndarray) and val.dtype == object:
            maybe = list(val)
            if len(maybe) and isinstance(maybe[0], dict):
                return maybe

    raise ValueError("Could not unpack subject results from npz.")


def load_msaa_results(analysis_type, fit_scope, K):
    path = find_msaa_npz(analysis_type, fit_scope, K)
    if path is None:
        return None, None

    d = load_npz_as_dict(path)
    results_subj = unpack_results_subj(d)
    return results_subj, path


def get_spatial_vector(results_subj, analysis_type, k):
    """
    Returns node-level vector for archetype k.

    spatial AA:
        node-level spatial weights are in S[k, :]

    temporal AA:
        node-level spatial weights are in sXC[:, k]
    """

    vectors = []

    for sub in results_subj:
        sXC = np.asarray(sub["sXC"], dtype=float)
        S = np.asarray(sub["S"], dtype=float)

        if analysis_type == "spatial":
            v = S[int(k), :]
        elif analysis_type == "temporal":
            v = sXC[:, int(k)]
        else:
            raise ValueError("analysis_type must be spatial or temporal")

        vectors.append(v)

    return np.nanmean(np.vstack(vectors), axis=0)


def network_mass_summary(vector, network_labels):
    v = np.asarray(vector, dtype=float)
    labels = np.asarray(network_labels)

    if len(v) != len(labels):
        raise ValueError(f"Vector length {len(v)} != labels length {len(labels)}")

    w = np.abs(v)
    total = np.sum(w)

    rows = []

    for lab in sorted(pd.unique(labels)):
        idx = labels == lab
        network = NETWORK_NAME_MAP[int(lab)]

        mass = np.sum(w[idx])
        expected = np.sum(idx) / len(labels)

        rows.append({
            "network_id": int(lab),
            "network": network,
            "n_nodes": int(np.sum(idx)),
            "expected_fraction_by_nodes": float(expected),
            "mass": float(mass),
            "mass_fraction": float(mass / total) if total > 0 else np.nan,
            "mass_minus_expected": float((mass / total) - expected) if total > 0 else np.nan,
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Load / build network labels if needed
# ------------------------------------------------------------

if "network_labels" not in globals():

    try:
        from nilearn.input_data import NiftiMasker
        import nibabel as nib
    except Exception as e:
        raise ImportError(
            "network_labels are not already defined, and nilearn/nibabel could not be imported. "
            "Run the network localization notebook first or install nilearn/nibabel."
        ) from e

    posterior = loadmat(POSTERIOR_MAT)
    centers = np.asarray(posterior["posterior"]["centers"][0][0][0][0][0], dtype=float)
    widths = np.asarray(list(posterior["posterior"]["widths"][0][0][0][0][0][:, 0].T), dtype=float).ravel()

    def fullfact(dims):
        vals = np.asmatrix(range(1, dims[0] + 1)).T
        if len(dims) == 1:
            return vals
        aftervals = np.asmatrix(fullfact(dims[1:]))
        inds = np.asmatrix(np.zeros((np.prod(dims), len(dims))))
        row = 0
        for i in range(aftervals.shape[0]):
            inds[row:(row + len(vals)), 0] = vals
            inds[row:(row + len(vals)), 1:] = np.tile(aftervals[i, :], (len(vals), 1))
            row += len(vals)
        return inds

    def nii2cmu(nifti_file, mask_file=None):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            img = nib.load(nifti_file) if type(nifti_file) == str else nifti_file
            mask = NiftiMasker(mask_strategy="background")
            mask.fit(nifti_file if mask_file is None else mask_file)
            S = img.get_sform()
            Y = np.float32(mask.transform(nifti_file)).copy()
            vmask = np.nonzero(
                np.array(
                    np.reshape(mask.mask_img_.dataobj, (1, np.prod(mask.mask_img_.shape)), order="C")
                )
            )[1]
            vox_coords = fullfact(img.shape[0:3])[vmask, ::-1] - 1
            R = np.array(np.dot(vox_coords, S[0:3, 0:3])) + S[:3, 3]
            return {"Y": Y, "R": R}

    def rbf(R, center, width):
        return np.exp(-np.sum((R - center) ** 2, axis=1) / width)

    key = pd.read_csv(
        SCHAEFER_TXT,
        sep="\t",
        header=None,
        names=["id", "name", "x", "y", "z", "t"],
    ).drop("t", axis=1)

    key["network"] = key["name"].apply(lambda x: LOOKUP_TABLE[x.split("_")[2]])
    key["code"] = key["network"].apply(lambda x: NETWORK_CODES[x])
    key.set_index("id", inplace=True)
    key.loc[0, "code"] = 0

    networks_cmu = nii2cmu(SCHAEFER_NII)
    networks_cmu["Y"] = np.atleast_2d(
        np.array([key.loc[i, "code"] for i in networks_cmu["Y"]]).astype(float)
    )

    labels = []
    for c, width in zip(centers, widths):
        r = rbf(networks_cmu["R"], c, width)
        label_weights = [
            sum(r[networks_cmu["Y"].ravel() == i])
            for i in range(1, len(NETWORK_CODES) + 1)
        ]
        labels.append(np.argmax(label_weights) + 1)

    network_labels = np.asarray(labels)

print("network_labels loaded:", len(network_labels))

# ------------------------------------------------------------
# Load top-m selected archetypes
# ------------------------------------------------------------

topm_path = Path(
    DECODING_DIR_TEMPLATE.format(
        analysis_type=BEST_ANALYSIS_TYPE,
        fit_scope=FIT_SCOPE,
    )
) / "topm_summary.csv"

if not topm_path.exists():
    raise FileNotFoundError(f"Could not find {topm_path}")

topm_df_for_networks = pd.read_csv(topm_path)

if "selected_archetype_list" not in topm_df_for_networks.columns:
    if "selected_archetypes" not in topm_df_for_networks.columns:
        raise ValueError("topm_summary.csv must contain selected_archetypes.")
    topm_df_for_networks["selected_archetype_list"] = topm_df_for_networks["selected_archetypes"].apply(parse_selected_archetypes)
else:
    topm_df_for_networks["selected_archetype_list"] = topm_df_for_networks["selected_archetype_list"].apply(parse_selected_archetypes)

topm_df_for_networks["analysis_type"] = topm_df_for_networks.get("analysis_type", BEST_ANALYSIS_TYPE)
topm_df_for_networks["fit_scope"] = topm_df_for_networks.get("fit_scope", FIT_SCOPE)
topm_df_for_networks["condition"] = topm_df_for_networks["condition"].astype(str)
topm_df_for_networks["K"] = topm_df_for_networks["K"].astype(int)
topm_df_for_networks["top_m"] = topm_df_for_networks["top_m"].astype(int)

print("Loaded top-m table:", topm_path)

# ------------------------------------------------------------
# Select best K,m configurations from joint_df
# ------------------------------------------------------------

best_configs = joint_df[
    (joint_df["analysis_type"] == BEST_ANALYSIS_TYPE)
    & (joint_df["condition"] == BEST_CONDITION)
    & (joint_df["cluster_metric"] == BEST_CLUSTER_METRIC)
].copy()

if BEST_MIN_TOP_M is not None:
    best_configs = best_configs[best_configs["top_m"] >= BEST_MIN_TOP_M]

if BEST_MAX_TOP_M is not None:
    best_configs = best_configs[best_configs["top_m"] <= BEST_MAX_TOP_M]

best_configs = (
    best_configs
    .sort_values("joint_zsum", ascending=False)
    .head(BEST_TOP_N_CONFIGS)
    .copy()
)

best_configs["config_label"] = best_configs.apply(
    lambda r: f"K={int(r['K'])}, m={int(r['top_m'])}",
    axis=1,
)

print("Best configurations:")
display(
    best_configs[
        [
            "analysis_type",
            "condition",
            "K",
            "top_m",
            "decode_z",
            "cluster_effect_z",
            "joint_zsum",
            "config_label",
        ]
    ]
)

# ------------------------------------------------------------
# Build network composition for only the selected best configs
# ------------------------------------------------------------

loaded_cache = {}
rows = []

for _, cfg in best_configs.iterrows():

    K = int(cfg["K"])
    top_m = int(cfg["top_m"])
    condition = str(cfg["condition"])

    selected_row = topm_df_for_networks[
        (topm_df_for_networks["K"] == K)
        & (topm_df_for_networks["top_m"] == top_m)
        & (topm_df_for_networks["condition"] == condition)
    ]

    if len(selected_row) == 0:
        print(f"Missing selected archetypes for K={K}, top_m={top_m}, condition={condition}")
        continue

    selected_archetypes = selected_row.iloc[0]["selected_archetype_list"]

    if len(selected_archetypes) == 0:
        print(f"Empty selected archetypes for K={K}, top_m={top_m}, condition={condition}")
        continue

    cache_key = (BEST_ANALYSIS_TYPE, K)
    if cache_key not in loaded_cache:
        results_subj, npz_path = load_msaa_results(BEST_ANALYSIS_TYPE, FIT_SCOPE, K)
        if results_subj is None:
            continue
        loaded_cache[cache_key] = results_subj
    else:
        results_subj = loaded_cache[cache_key]

    for rank_in_subset, arch in enumerate(selected_archetypes, start=1):

        v = get_spatial_vector(results_subj, BEST_ANALYSIS_TYPE, arch)

        mass_df = network_mass_summary(v, network_labels)

        for _, r in mass_df.iterrows():

            rows.append({
                "analysis_type": BEST_ANALYSIS_TYPE,
                "condition": condition,
                "K": K,
                "top_m": top_m,
                "archetype": int(arch),
                "rank_in_subset": rank_in_subset,
                "config_label": cfg["config_label"],
                "decode_z": float(cfg["decode_z"]),
                "cluster_effect_z": float(cfg["cluster_effect_z"]),
                "joint_zsum": float(cfg["joint_zsum"]),
                "network": r["network"],
                "network_id": int(r["network_id"]),
                "n_nodes": int(r["n_nodes"]),
                "expected_fraction_by_nodes": float(r["expected_fraction_by_nodes"]),
                "mass_fraction": float(r["mass_fraction"]),
                "mass_minus_expected": float(r["mass_minus_expected"]),
            })

best_combo_network_df = pd.DataFrame(rows)

print("best_combo_network_df shape:", best_combo_network_df.shape)
display(best_combo_network_df.head())

# ------------------------------------------------------------
# Summarize across best configurations
# ------------------------------------------------------------

# positive weights based on joint_zsum
min_joint = best_combo_network_df["joint_zsum"].min()
best_combo_network_df["joint_weight"] = best_combo_network_df["joint_zsum"] - min_joint + 1e-6

summary_rows = []

for network in NETWORK_ORDER:

    s = best_combo_network_df[best_combo_network_df["network"] == network].copy()

    if len(s) == 0:
        continue

    weights = s["joint_weight"].to_numpy(dtype=float)

    summary_rows.append({
        "network": network,
        "mean_mass_fraction": s["mass_fraction"].mean(),
        "weighted_mass_fraction": np.average(s["mass_fraction"], weights=weights),
        "mean_mass_minus_expected": s["mass_minus_expected"].mean(),
        "weighted_mass_minus_expected": np.average(s["mass_minus_expected"], weights=weights),
        "mean_expected_fraction_by_nodes": s["expected_fraction_by_nodes"].mean(),
    })

best_combo_network_summary_df = pd.DataFrame(summary_rows)

print("Summary across best configurations:")
display(
    best_combo_network_summary_df
    .sort_values("weighted_mass_fraction", ascending=False)
)

# ------------------------------------------------------------
# Plot 1: weighted network representation in best K,m combinations
# ------------------------------------------------------------

plot_df = (
    best_combo_network_summary_df
    .set_index("network")
    .reindex(NETWORK_ORDER)
    .reset_index()
)

x = np.arange(len(plot_df))
bar_colors = [NETWORK_COLORS[n] for n in plot_df["network"]]

fig, ax = plt.subplots(figsize=(7.5, 4.3))

ax.bar(
    x,
    plot_df["weighted_mass_fraction"],
    color=bar_colors,
    edgecolor="black",
    linewidth=0.7,
)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["network"], rotation=35, ha="right")

ax.set_ylabel("Weighted mean mass fraction")
ax.set_title(
    f"Network representation in best joint decoding-clustering configurations\n"
    f"{BEST_CONDITION}, top {BEST_TOP_N_CONFIGS} configurations"
)

plt.tight_layout()

save_current_fig(
    f"best_joint_configs_weighted_network_mass_"
    f"{BEST_ANALYSIS_TYPE}_{BEST_CONDITION}_top{BEST_TOP_N_CONFIGS}"
)

plt.show()
plt.close()

# ------------------------------------------------------------
# Plot 2: weighted over/under-representation relative to network size
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.5, 4.3))

ax.bar(
    x,
    plot_df["weighted_mass_minus_expected"],
    color=bar_colors,
    edgecolor="black",
    linewidth=0.7,
)

ax.axhline(0, color="gray", linestyle="--", linewidth=1)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["network"], rotation=35, ha="right")

ax.set_ylabel("Weighted mass fraction minus expected")
ax.set_title(
    f"Network over-representation in best joint decoding-clustering configurations\n"
    f"{BEST_CONDITION}, top {BEST_TOP_N_CONFIGS} configurations"
)

plt.tight_layout()

save_current_fig(
    f"best_joint_configs_network_overrepresentation_"
    f"{BEST_ANALYSIS_TYPE}_{BEST_CONDITION}_top{BEST_TOP_N_CONFIGS}"
)

plt.show()
plt.close()

# ------------------------------------------------------------
# Plot 3: heatmap of network composition across best K,m configs
# ------------------------------------------------------------

config_order = best_configs["config_label"].tolist()

mat = (
    best_combo_network_df
    .pivot_table(
        index="network",
        columns="config_label",
        values="mass_fraction",
        aggfunc="mean",
    )
    .reindex(index=NETWORK_ORDER, columns=config_order)
)

fig, ax = plt.subplots(figsize=(max(9, 0.42 * len(config_order)), 4.6))

im = ax.imshow(
    mat.values,
    aspect="auto",
    interpolation="nearest",
    cmap="viridis",
)

ax.set_yticks(np.arange(len(mat.index)))
ax.set_yticklabels(mat.index)

ax.set_xticks(np.arange(len(mat.columns)))
ax.set_xticklabels(mat.columns, rotation=60, ha="right")

ax.set_ylabel("Network")
ax.set_xlabel("Best-performing K,m configuration")

ax.set_title(
    f"Network composition across best joint decoding-clustering configurations\n"
    f"{BEST_CONDITION}"
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Mean mass fraction across selected archetypes")

plt.tight_layout()

save_current_fig(
    f"best_joint_configs_network_composition_heatmap_"
    f"{BEST_ANALYSIS_TYPE}_{BEST_CONDITION}_top{BEST_TOP_N_CONFIGS}"
)

plt.show()
plt.close()

# ------------------------------------------------------------
# Save tables
# ------------------------------------------------------------

out1 = FIG_DIR / f"best_joint_configs_network_composition_{BEST_ANALYSIS_TYPE}_{BEST_CONDITION}.csv"
out2 = FIG_DIR / f"best_joint_configs_network_summary_{BEST_ANALYSIS_TYPE}_{BEST_CONDITION}.csv"

best_combo_network_df.to_csv(out1, index=False)
best_combo_network_summary_df.to_csv(out2, index=False)

print("Saved:", out1)
print("Saved:", out2)

In [ ]:
# ============================================================
# NETWORK OVERREPRESENTATION IN BEST K,m COMBINATIONS
# ============================================================

summary = (
    best_combo_network_df
    .groupby("network")
    .agg({
        "mass_fraction": "mean",
        "expected_fraction_by_nodes": "mean",
    })
    .reset_index()
)

summary["enrichment_ratio"] = (
    summary["mass_fraction"]
    /
    summary["expected_fraction_by_nodes"]
)

summary = (
    summary
    .set_index("network")
    .reindex(NETWORK_ORDER)
    .reset_index()
)

display(summary)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(13,4),
    sharey=True
)

x = np.arange(len(summary))

colors = [NETWORK_COLORS[n] for n in summary["network"]]

# ------------------------------------------------------------
# Raw observed mass
# ------------------------------------------------------------

axes[0].bar(
    x,
    summary["mass_fraction"],
    color=colors,
    edgecolor="black",
)

axes[0].set_title("Observed")
axes[0].set_ylabel("Mean mass fraction")

# ------------------------------------------------------------
# Expected by network size
# ------------------------------------------------------------

axes[1].bar(
    x,
    summary["expected_fraction_by_nodes"],
    color=colors,
    edgecolor="black",
)

axes[1].set_title("Expected\n(from node count)")

# ------------------------------------------------------------
# Enrichment ratio
# ------------------------------------------------------------

axes[2].bar(
    x,
    summary["enrichment_ratio"],
    color=colors,
    edgecolor="black",
)

axes[2].axhline(
    1.0,
    color="black",
    linestyle="--",
    linewidth=1,
)

axes[2].set_title("Observed / Expected")
axes[2].set_ylabel("Enrichment ratio")

# ------------------------------------------------------------

for ax in axes:

    ax.set_xticks(x)
    ax.set_xticklabels(
        summary["network"],
        rotation=40,
        ha="right",
    )

plt.suptitle(
    f"Network representation among best joint decoding-clustering configurations\n"
    f"{BEST_ANALYSIS_TYPE}, {BEST_CONDITION}"
)

plt.tight_layout()

save_current_fig(
    f"network_enrichment_ratio_best_configs_"
    f"{BEST_ANALYSIS_TYPE}_{BEST_CONDITION}"
)

plt.show()
plt.close()

In [ ]:
# ============================================================
# NETWORK ENRICHMENT WITH UNCERTAINTY
#
# Unit of uncertainty:
#   best-performing K,m configurations
#
# Requires:
#   best_combo_network_df
#   NETWORK_ORDER
#   NETWORK_COLORS
#   BEST_ANALYSIS_TYPE
#   BEST_CONDITION
#   save_current_fig
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

N_BOOT = 5000
CI_LOW = 2.5
CI_HIGH = 97.5
RANDOM_SEED = 0

rng = np.random.default_rng(RANDOM_SEED)

# ------------------------------------------------------------
# Collapse to one value per configuration x network
# ------------------------------------------------------------

config_network_df = (
    best_combo_network_df
    .groupby(["config_label", "network"], as_index=False)
    .agg(
        mass_fraction=("mass_fraction", "mean"),
        expected_fraction_by_nodes=("expected_fraction_by_nodes", "mean"),
        joint_zsum=("joint_zsum", "mean"),
    )
)

config_network_df["enrichment_ratio"] = (
    config_network_df["mass_fraction"]
    /
    config_network_df["expected_fraction_by_nodes"]
)

config_network_df["mass_minus_expected"] = (
    config_network_df["mass_fraction"]
    -
    config_network_df["expected_fraction_by_nodes"]
)

# ------------------------------------------------------------
# Bootstrap CIs across configurations
# ------------------------------------------------------------

summary_rows = []

for network in NETWORK_ORDER:

    s = config_network_df[
        config_network_df["network"] == network
    ].copy()

    vals = s["enrichment_ratio"].to_numpy(dtype=float)
    vals = vals[np.isfinite(vals)]

    vals_diff = s["mass_minus_expected"].to_numpy(dtype=float)
    vals_diff = vals_diff[np.isfinite(vals_diff)]

    if len(vals) == 0:
        continue

    boot = []
    boot_diff = []

    for _ in range(N_BOOT):
        idx = rng.integers(0, len(vals), size=len(vals))
        boot.append(np.mean(vals[idx]))

        idx2 = rng.integers(0, len(vals_diff), size=len(vals_diff))
        boot_diff.append(np.mean(vals_diff[idx2]))

    boot = np.asarray(boot)
    boot_diff = np.asarray(boot_diff)

    summary_rows.append({
        "network": network,
        "n_configs": len(vals),

        "mean_enrichment_ratio": np.mean(vals),
        "ci_low_enrichment_ratio": np.percentile(boot, CI_LOW),
        "ci_high_enrichment_ratio": np.percentile(boot, CI_HIGH),

        "mean_mass_minus_expected": np.mean(vals_diff),
        "ci_low_mass_minus_expected": np.percentile(boot_diff, CI_LOW),
        "ci_high_mass_minus_expected": np.percentile(boot_diff, CI_HIGH),
    })

enrichment_uncertainty_df = pd.DataFrame(summary_rows)

enrichment_uncertainty_df = (
    enrichment_uncertainty_df
    .set_index("network")
    .reindex(NETWORK_ORDER)
    .reset_index()
)

display(enrichment_uncertainty_df)

# ------------------------------------------------------------
# Plot 1: enrichment ratio with bootstrap 95% CI
# ------------------------------------------------------------

plot_df = enrichment_uncertainty_df.copy()
x = np.arange(len(plot_df))
colors = [NETWORK_COLORS[n] for n in plot_df["network"]]

y = plot_df["mean_enrichment_ratio"].to_numpy(dtype=float)
yerr = np.vstack([
    y - plot_df["ci_low_enrichment_ratio"].to_numpy(dtype=float),
    plot_df["ci_high_enrichment_ratio"].to_numpy(dtype=float) - y,
])

fig, ax = plt.subplots(figsize=(7.8, 4.6))

ax.bar(
    x,
    y,
    yerr=yerr,
    capsize=4,
    color=colors,
    edgecolor="black",
    linewidth=0.7,
    alpha=0.9,
)

# overlay individual config-level points
for i, network in enumerate(plot_df["network"]):
    vals = config_network_df.loc[
        config_network_df["network"] == network,
        "enrichment_ratio"
    ].to_numpy(dtype=float)

    jitter = rng.normal(0, 0.045, size=len(vals))

    ax.scatter(
        np.full(len(vals), i) + jitter,
        vals,
        s=18,
        alpha=0.35,
        color="black",
        linewidth=0,
    )

ax.axhline(
    1.0,
    color="black",
    linestyle="--",
    linewidth=1,
)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["network"], rotation=35, ha="right")

ax.set_ylabel("Observed / expected network mass")
ax.set_title(
    "Network enrichment in best joint decoding-clustering configurations\n"
    f"{BEST_ANALYSIS_TYPE}, {BEST_CONDITION}; error bars = bootstrap 95% CI"
)

plt.tight_layout()

save_current_fig(
    f"network_enrichment_ratio_bootstrapCI_"
    f"{BEST_ANALYSIS_TYPE}_{BEST_CONDITION}"
)

plt.show()
plt.close()

# ------------------------------------------------------------
# Plot 2: mass minus expected with bootstrap 95% CI
# ------------------------------------------------------------

y = plot_df["mean_mass_minus_expected"].to_numpy(dtype=float)
yerr = np.vstack([
    y - plot_df["ci_low_mass_minus_expected"].to_numpy(dtype=float),
    plot_df["ci_high_mass_minus_expected"].to_numpy(dtype=float) - y,
])

fig, ax = plt.subplots(figsize=(7.8, 4.6))

ax.bar(
    x,
    y,
    yerr=yerr,
    capsize=4,
    color=colors,
    edgecolor="black",
    linewidth=0.7,
    alpha=0.9,
)

for i, network in enumerate(plot_df["network"]):
    vals = config_network_df.loc[
        config_network_df["network"] == network,
        "mass_minus_expected"
    ].to_numpy(dtype=float)

    jitter = rng.normal(0, 0.045, size=len(vals))

    ax.scatter(
        np.full(len(vals), i) + jitter,
        vals,
        s=18,
        alpha=0.35,
        color="black",
        linewidth=0,
    )

ax.axhline(
    0.0,
    color="black",
    linestyle="--",
    linewidth=1,
)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["network"], rotation=35, ha="right")

ax.set_ylabel("Observed mass fraction - expected fraction")
ax.set_title(
    "Network overrepresentation in best joint decoding-clustering configurations\n"
    f"{BEST_ANALYSIS_TYPE}, {BEST_CONDITION}; error bars = bootstrap 95% CI"
)

plt.tight_layout()

save_current_fig(
    f"network_mass_minus_expected_bootstrapCI_"
    f"{BEST_ANALYSIS_TYPE}_{BEST_CONDITION}"
)

plt.show()
plt.close()

In [ ]:
# ============================================================
# EXPECTED -> OBSERVED -> ENRICHMENT
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(13,4),
    sharex=False
)

plot_df = enrichment_uncertainty_df.copy()

x = np.arange(len(plot_df))

colors = [
    NETWORK_COLORS[n]
    for n in plot_df["network"]
]

# --------------------------------------------------------
# Expected
# --------------------------------------------------------

expected = (
    config_network_df
    .groupby("network")["expected_fraction_by_nodes"]
    .mean()
    .reindex(NETWORK_ORDER)
)

axes[0].bar(
    x,
    expected.values,
    color=colors,
    edgecolor="black",
)

axes[0].set_title("Expected\n(network size)")
axes[0].set_ylabel("Fraction of nodes")

# --------------------------------------------------------
# Observed
# --------------------------------------------------------

observed = (
    config_network_df
    .groupby("network")["mass_fraction"]
    .mean()
    .reindex(NETWORK_ORDER)
)

axes[1].bar(
    x,
    observed.values,
    color=colors,
    edgecolor="black",
)

axes[1].set_title("Observed\n(best K,m combinations)")
axes[1].set_ylabel("Mass fraction")

# --------------------------------------------------------
# Enrichment
# --------------------------------------------------------

y = plot_df["mean_mass_minus_expected"].values

yerr = np.vstack([
    y - plot_df["ci_low_mass_minus_expected"].values,
    plot_df["ci_high_mass_minus_expected"].values - y,
])

axes[2].bar(
    x,
    y,
    yerr=yerr,
    capsize=4,
    color=colors,
    edgecolor="black",
)

axes[2].axhline(
    0,
    color="black",
    linestyle="--",
)

axes[2].set_title("Observed - Expected")
axes[2].set_ylabel("Overrepresentation")

# --------------------------------------------------------

for ax in axes:

    ax.set_xticks(x)

    ax.set_xticklabels(
        plot_df["network"],
        rotation=35,
        ha="right",
    )

plt.suptitle(
    "Network representation in best joint decoding-clustering configurations"
)

plt.tight_layout()

save_current_fig(
    f"best_joint_configs_network_composition_heatmap_"
    f"{BEST_ANALYSIS_TYPE}_{BEST_CONDITION}_top{BEST_TOP_N_CONFIGS}"
)

plt.show()

In [ ]:
# ============================================================
# SAVE OUTPUTS
# ============================================================

out_joint = FIG_DIR / f"joint_decoding_clustering_long_{FIT_SCOPE}.csv"
out_consensus = FIG_DIR / f"joint_decoding_clustering_consensus_{FIT_SCOPE}.csv"

joint_df.to_csv(out_joint, index=False)
joint_consensus_df.to_csv(out_consensus, index=False)

print("Saved:", out_joint)
print("Saved:", out_consensus)


## Interpretation guide

The main outputs are:

```python
joint_df
joint_consensus_df
```

Important columns:

- `decode_mean`: top-\(m\) decoding score;
- `cluster_observed_minus_random`: clustering advantage relative to random archetype subsets;
- `cluster_z_vs_random`: random-subset validation strength;
- `decode_z`: relative decoding strength;
- `cluster_effect_z`: relative clustering-effect strength;
- `joint_zsum`: main joint score;
- `joint_percentile_geom`: robust percentile-based joint score.

A strong joint regime should have high decoding, positive clustering advantage, positive `cluster_z_vs_random`, and high `joint_zsum`.
